In [1]:
# Se define el entrenamiento para repetir el flujo sin omitir pasos.

def train_estimator():

    import os
    from pathlib import Path

    import pandas as pd
    from sklearn.feature_extraction.text import TfidfTransformer
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.feature_extraction.text import CountVectorizer

    import pickle

    SENTENCES = "../data/sentences.csv.gz"
    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

    sentences_path = Path(SENTENCES)
    assert sentences_path.is_file(), f"No se encontró el archivo de datos: {sentences_path}"
    assert sentences_path.stat().st_size > 0, f"El archivo de datos está vacío: {sentences_path}"

    dataframe = pd.read_csv(
        SENTENCES,
        index_col=False,
        compression="gzip",
    )

    data = dataframe.phrase
    target = dataframe.target

    X_train, _, y_train, _ = train_test_split(
        data,
        target,
        test_size=0.3,
        shuffle=False,
    )

    #
    # Modelo de regresión logística
    #
    vectorizer = CountVectorizer(
        lowercase=True,
        analyzer="word",
        token_pattern=r"\b[a-zA-Z]\w+\b",
        stop_words="english",
    )

    transformer = TfidfTransformer()

    lr_estimator = Pipeline(
        steps=[
            ("vectorizer", vectorizer),
            ("transformer", transformer),
            ("estimator", LogisticRegression(max_iter=1000)),
        ],
        verbose=False,
    )

    lr_estimator.fit(X_train, y_train)

    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    with open(ESTIMATOR_PATH, "wb") as file:
        pickle.dump(lr_estimator, file)


train_estimator()

In [2]:
# Se define el uso del estimador ya almacenado.

def use_estimator():

    import pickle
    from pathlib import Path

    import pandas as pd

    SENTENCES = "../data/sentences.csv.gz"
    OUTPUT_FOLDER = "../submission"
    ESTIMATOR_PATH = f"{OUTPUT_FOLDER}/estimator.pkl"

    sentences_path = Path(SENTENCES)
    assert sentences_path.is_file(), f"No se encontró el archivo de datos: {sentences_path}"
    assert sentences_path.stat().st_size > 0, f"El archivo de datos está vacío: {sentences_path}"

    dataframe = pd.read_csv(
        SENTENCES,
        index_col=False,
        compression="gzip",
    )

    data = dataframe.phrase

    with open(ESTIMATOR_PATH, "rb") as file:
        estimator = pickle.load(file)

    prediction = estimator.predict(data)

    return prediction


use_estimator()

array(['neutral', 'positive', 'positive', ..., 'neutral', 'positive',
       'neutral'], dtype=object)